# Distill — entraîner un SLM spécialisé sur un GPU gratuit

Ce notebook exécute le pipeline complet de bout en bout sur le GPU gratuit de Colab
ou de Kaggle : génération des données, critique, fine-tuning LoRA, évaluation face au
professeur, export GGUF et Modelfile Ollama.

**Durée** : environ 45 à 90 minutes pour un élève de 1.5B et 3 000 exemples.
**Coût** : le GPU est gratuit ; seul le professeur est facturé (budget plafonné dans `task.yaml`).

> Un T4 gratuit suffit pour les élèves jusqu'à 1.5B en QLoRA 4 bits. Pour un 3B, prenez un
> L4 ou un A100, ou réduisez `student.max_seq_length`.

Projet : [github.com/Abdelilah04116/distill](https://github.com/Abdelilah04116/distill) — Abdelilah OURTI, AI Engineer.

## 1. Vérifier le GPU

Si cette cellule échoue, activez un GPU : *Exécution → Modifier le type d'exécution → T4*.

In [ ]:
import subprocess, sys

print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'],
                     capture_output=True, text=True).stdout or 'AUCUN GPU DETECTE')
print('Python', sys.version.split()[0])

## 2. Installer Distill et ses dépendances d'entraînement

Unsloth est optionnel mais divise par deux le temps d'entraînement et la VRAM utilisée.
S'il refuse de s'installer, le pipeline bascule tout seul sur Transformers + PEFT + TRL.

In [ ]:
%%capture
# Le dépôt, avec les extras entraînement, graphe et dédup.
!pip install -q 'distill-slm[train,graph,dedup] @ git+https://github.com/Abdelilah04116/distill.git'

# Unsloth : accélère l'entraînement. L'échec n'est pas bloquant.
!pip install -q unsloth || echo 'Unsloth indisponible, repli sur TRL'

In [ ]:
import distill

print('Distill', distill.__version__)
!distill version

## 3. Compiler llama.cpp

Nécessaire uniquement pour l'export GGUF. Comptez deux à trois minutes.

In [ ]:
%%capture
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
!cd /content/llama.cpp && cmake -B build -DLLAMA_CURL=OFF && cmake --build build --config Release -j4
!pip install -q -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt

In [ ]:
import os
from pathlib import Path

os.environ['LLAMA_CPP_DIR'] = '/content/llama.cpp'
quantize = list(Path('/content/llama.cpp/build').rglob('llama-quantize'))
print('llama-quantize :', quantize[0] if quantize else 'INTROUVABLE — recompilez la cellule precedente')

## 4. Le professeur

Deux options.

**A. API Anthropic** — la plus simple. La clé n'est jamais écrite dans le notebook.

**B. Modèle local via vLLM** — aucune donnée ne quitte la machine, mais il faut un GPU
nettement plus gros. Décommentez le bloc B et adaptez `teacher` dans le `task.yaml`.

In [ ]:
from getpass import getpass
import os

# --- A. professeur via API ---
os.environ['ANTHROPIC_API_KEY'] = getpass('Clé API Anthropic (collée, non affichée) : ')

# --- B. professeur local via vLLM (decommenter, et adapter teacher dans task.yaml) ---
# !pip install -q vllm
# import subprocess
# subprocess.Popen(['vllm', 'serve', 'Qwen/Qwen2.5-7B-Instruct', '--port', '8000'])
# os.environ['DISTILL_TEACHER_API_KEY'] = 'local'

## 5. La tâche

On récupère l'exemple livré avec le dépôt : triage d'emails clients en français,
24 seeds et 201 exemples de test.

**Pour votre propre tâche** : remplacez `task.yaml`, `seeds.jsonl` et `test_set.jsonl`.
Le reste du notebook ne change pas.

In [ ]:
!git clone --depth 1 https://github.com/Abdelilah04116/distill.git /content/distill-repo
!cp -r /content/distill-repo/examples /content/examples
!ls -l /content/examples/email-triage

In [ ]:
TASK = '/content/examples/email-triage/task.yaml'

# Vérifie le schéma, les seeds et le test set avant de dépenser quoi que ce soit.
!distill validate --task $TASK

## 6. Estimer le coût avant de lancer

Le plafond vit dans `task.yaml` (`budget.max_cost_usd`). Dépassé, le run s'arrête
proprement et conserve ce qui a déjà été produit.

In [ ]:
!distill estimate --task $TASK
!distill run --task $TASK --dry-run

## 7. Le run complet

Une commande. Génération, critique, entraînement LoRA, évaluation face au professeur,
boucle d'amélioration si besoin, export GGUF et Modelfile, rapport HTML.

Pour un premier essai moins cher, réduisez le volume : `--max-examples 500`.

In [ ]:
# Astuce : --max-examples 500 pour un premier run rapide et bon marché.
!distill run --task $TASK --runs-dir /content/runs --yes

## 8. Le tableau de résultats

Le rapport s'affiche directement dans le notebook. C'est le livrable à publier :
il contient le tableau comparatif, la courbe de loss, et les erreurs réelles de l'élève.

In [ ]:
import json
from pathlib import Path
from IPython.display import HTML, display

runs = sorted(Path('/content/runs/email-triage').glob('2*'))
run = runs[-1]
print('run :', run)

metrics = json.loads((run / 'eval' / 'metrics.json').read_text(encoding='utf-8'))
for model in metrics['models']:
    print(f"{model['name']:<40} F1 {model['macro_f1']:.3f}  JSON {model['json_valid_rate']:.1%}"
          f"  {model['latency_mean_s'] * 1000:.0f} ms  {model['cost_per_1k_usd']:.4f} $/1k")
print()
print(f"ratio eleve / professeur : {metrics['quality_ratio']:.1%}")
print(f"acceleration : {metrics['speedup']:.1f}x   economie : {metrics['cost_factor']:.1f}x")

In [ ]:
display(HTML((run / 'report.html').read_text(encoding='utf-8')))

## 9. Essayer le modèle ici même

Ollama tourne aussi dans le notebook : c'est exactement l'artefact quantifié que vous
livrerez, pas les poids restés sur le GPU.

In [ ]:
%%capture
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(['ollama', 'serve'])
time.sleep(5)

In [ ]:
!cd {run}/export && ollama create distill-email-triage -f Modelfile
!ollama list

In [ ]:
EMAIL = "Bonjour, je n'ai toujours pas recu ma commande passee le 3, et personne ne repond au telephone. C'est inadmissible."

!ollama run distill-email-triage "{EMAIL}"

In [ ]:
# Réévaluation sur le GGUF réellement livré : les chiffres les plus honnêtes à publier.
!distill eval --task $TASK --runs-dir /content/runs --backend ollama

## 10. Récupérer les artefacts

Le GGUF, le Modelfile et le rapport. Gardez aussi `data/train.jsonl` et `data/test.jsonl` :
sans eux, le run n'est pas reproductible.

In [ ]:
import shutil

archive = shutil.make_archive('/content/distill-email-triage', 'zip', run)
print('archive :', archive, f"({Path(archive).stat().st_size / 1e6:.0f} Mo)")

try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Hors Colab : récupérez le fichier depuis le panneau de droite.')

## 11. Publier sur Hugging Face (optionnel)

`distill publish` envoie le GGUF, le Modelfile et une carte de modèle pré-remplie avec
les chiffres mesurés, y compris les limites et les cas d'échec.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()
# !distill publish --task $TASK --runs-dir /content/runs --repo votre-compte/distill-email-triage

---

**Avant de publier le moindre chiffre** : relisez `data/test.jsonl` à la main. Le test set
livré avec le dépôt a été rédigé pour la démonstration, pas validé par un humain. C'est
sur ce point qu'un benchmark se fait démonter, et c'est le seul que personne ne peut
automatiser à votre place.